# 11b — Batch Normalization (PyTorch)

Corresponds to: Note 11 — Batch Normalization

**Libraries introduced in this notebook:** No new libraries — PyTorch introduced in Note 09.

**What this notebook demonstrates:**
1. `nn.BatchNorm1d` — matches the hand implementation from 11a
2. Training vs inference mode — running statistics and `model.eval()`


In [ ]:
import torch
import torch.nn as nn

print(f'PyTorch version: {torch.__version__}')


## Part 1: `nn.BatchNorm1d`

In 11a we implemented batch norm from scratch:

    mu_B  = mean(z)
    var_B = var(z)
    z_hat = (z - mu_B) / sqrt(var_B + eps)
    y     = gamma * z_hat + beta

`nn.BatchNorm1d` does the same thing. We verify it produces the same output as the worked example.

In [ ]:
# nn.BatchNorm1d expects input shape (N, C) — N samples, C features
# Our worked example is 4 samples, 1 feature
z = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

bn = nn.BatchNorm1d(num_features=1, eps=0.0, momentum=None)

# Set gamma=2.0, beta=1.0 to match the worked example
with torch.no_grad():
    bn.weight.fill_(2.0)   # gamma
    bn.bias.fill_(1.0)     # beta

bn.train()
y = bn(z)
print(f'y = {y.detach().flatten().numpy().round(3)}')
print('expected: [-1.684  0.106  1.894  3.684]')


The outputs match the hand calculation from Note 11 Part 2.

Note: PyTorch's `BatchNorm1d` uses Bessel's correction (divides by m−1) for the running variance estimate, but uses the biased variance (divides by m) for normalization — consistent with the note.

## Part 2: Training vs Inference Mode

Batch norm uses batch statistics during training and running statistics at inference.
We simulate several training batches to accumulate running statistics, then switch to eval mode.

In [ ]:
torch.manual_seed(0)
bn = nn.BatchNorm1d(num_features=4)

# Simulate training — pass several batches to accumulate running statistics
bn.train()
for _ in range(20):
    x_batch = torch.randn(8, 4)
    _ = bn(x_batch)

print(f'Running mean after training: {bn.running_mean.numpy().round(3)}')
print(f'Running var  after training: {bn.running_var.numpy().round(3)}')

# Single sample — training mode uses batch stats (meaningless for 1 sample)
x_single = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
bn.train()
y_train = bn(x_single)
print(f'\nTraining mode output (1 sample): {y_train.detach().numpy().round(3)}')
print('(batch stats from 1 sample — mean=input, var=0 — output is undefined/nan-prone)')

# Eval mode uses running statistics — deterministic and correct
bn.eval()
y_eval = bn(x_single)
print(f'Eval mode output    (1 sample): {y_eval.detach().numpy().round(3)}')
print('(running stats from training — stable and deterministic)')


## Summary

This notebook demonstrated:

1. **`nn.BatchNorm1d`** — produces the same output as the hand implementation in 11a. gamma and beta are `weight` and `bias`.
2. **Training vs inference mode** — `model.train()` uses batch statistics, `model.eval()` uses running statistics. Always call `model.eval()` before inference.
